<div style="text-align:center"><h2>Deep Learning Project</h2></div>

In [4]:
!pip install youtube_transcript_api google-api-python-client python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached httplib2-0.22.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached googleapis_common_protos-1.70.0-py3-none-any.whl.metadata (9.3 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
    --------------------------------------- 0.3/13.7 MB ? eta -:--:--
    --------------------------------------- 0.3/13.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.7 MB 1.4 MB/s

In [16]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from youtube_transcript_api.proxies import WebshareProxyConfig
from googleapiclient.discovery import build
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import requests
import time
import os
import re

load_dotenv("./.env", override=True)

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
url_sample = "https://www.youtube.com/"

### Youtube Data (Include Transcript) Retrieve

In [20]:
class YouTubeDataCollector:
    def __init__(self, channel_name, api_key=YOUTUBE_API_KEY):
        self.channel_name = channel_name
        print("Data Collector Initialized for channel: ", self.channel_name)
        self.channel_id = self.get_channel_id()
        self.api_key = api_key
        self.youtube = build('youtube', 'v3', developerKey=api_key)
        self.number_of_videos = 0
    
    def get_channel_id(self):
        url_request = url_sample + self.channel_name
        response = requests.get(url_request)
        if response.status_code == 200:
            data = response.text
            match = re.search(r'"key":"browse_id","value":"([^"]+)"', data)
            if match:
                value = match.group(1)
                self.chanel_id = value
            else:
                self.chanel_id = None
        return self.chanel_id

    def get_channel_stats(self):
        request = self.youtube.channels().list(part='snippet,contentDetails,statistics', id=self.channel_id)
        response = request.execute()
        return response['items']

    def get_playlist_id(self, channel_stats):
        playlist_id = channel_stats[0]['contentDetails']['relatedPlaylists']['uploads']
        return playlist_id

    def get_video_list(self, playlist_ID):
        video_list = []
        request = self.youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=playlist_ID,
            maxResults=50
        )
        next_page = True
        while next_page:
            response = request.execute()
            data = response['items']

            for video in data:
                video_id = video['contentDetails']['videoId']
                if video_id not in video_list:
                    video_list.append(video_id)

            if 'nextPageToken' in response:
                next_page = True
                request = self.youtube.playlistItems().list(
                    part="snippet,contentDetails",
                    playlistId=playlist_ID,
                    maxResults=50,
                    pageToken=response['nextPageToken']
                )
            else:
                next_page = False

        return video_list
    
    def get_full_transcript(self, video_id: str, languages: list = ['en']) -> str:
        try:
            transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
            #print(transcript)
            time.sleep(5)
            return "\n".join(item['text'] for item in transcript)
        except TranscriptsDisabled:
            return "Transcripts are disabled for this video."
        except NoTranscriptFound:
            return "No transcript available in the requested languages."

    def get_video_details(self, video_list):
        stats_list = []
        counter = 0
        for i in range(0, len(video_list), 50):
            request = self.youtube.videos().list(
                part="snippet,contentDetails,statistics",
                id=video_list[i:i + 50]
            )
            response = request.execute()

            for video in response['items']:
                video_id = video['id']
                title = video['snippet']['title']
                published = video['snippet']['publishedAt']
                description = video['snippet']['description']
                video_tags = video['snippet'].get('tags', [])
                video_default_language = video['snippet'].get('defaultAudioLanguage', '')
                video_duration = video['contentDetails'].get('duration', '')
                tag_count = len(video_tags)
                views_count = video['statistics'].get('viewCount', 0)
                dislikes_count = video['statistics'].get('dislikeCount', 0)
                likes_count = video['statistics'].get('likeCount', 0)
                comments_count = video['statistics'].get('commentCount', 0)
                transcript = video['contentDetails']['caption']#self.get_full_transcript(video_id, languages=[video_default_language])
                #print(video)

                stats_dictionary = dict(
                    video_id=video_id,
                    title=title,
                    published=published,
                    description=description,
                    video_tags=video_tags,
                    video_default_language=video_default_language,
                    video_duration=video_duration,
                    tag_count=tag_count,
                    views_count=views_count,
                    dislikes_count=dislikes_count,
                    likes_count=likes_count,
                    comments_count=comments_count,
                    transcript=transcript
                )

                stats_list.append(stats_dictionary)
            counter += 50
            print(f"Processed {counter} videos")

        return stats_list
    
    def collect_data(self):
        channel_stats = self.get_channel_stats()
        playlist_id = self.get_playlist_id(channel_stats)
        video_list = self.get_video_list(playlist_id)
        number_of_videos = len(video_list)
        self.number_of_videos = number_of_videos
        video_data_list = self.get_video_details(video_list[:50])
        return video_data_list


if __name__ == "__main__":
    list_of_channel_names = pd.read_csv('./channel_name.csv')
    for index, row in list_of_channel_names.iterrows():
        youtube_collector = YouTubeDataCollector(row['channel_name'])
        transformed_data = youtube_collector.collect_data()    

Data Collector Initialized for channel:  @TED
Processed 50 videos


In [21]:
transformed_data[0]

{'video_id': 'Dp_MbiueqjI',
 'title': 'Former VP Al Gore debunks “climate realism” and urges action as climate progress happens. #TEDTalks',
 'published': '2025-07-07T16:00:31Z',
 'description': 'In this urgent and hard-hitting talk, Nobel Laureate Al Gore thoroughly dismantles the fossil fuel industry’s narrative of "climate realism," contrasting their misleading claims with the remarkable advancements in renewable energy. Drawing on data showing clear signs of progress across the world, Gore makes a powerful case that we already have everything needed to solve the climate crisis — and reminds us of what the most valuable renewable resource actually is.',
 'video_tags': ['TEDTalk', 'TEDTalks', 'TED Talk', 'TED Talks', 'TED'],
 'video_default_language': 'en',
 'video_duration': 'PT1M34S',
 'tag_count': 5,
 'views_count': '8570',
 'dislikes_count': 0,
 'likes_count': '372',
 'comments_count': '18',
 'transcript': 'false'}

In [23]:
count = 0

for index, row in enumerate(transformed_data):
    if row['transcript'] == "true":
        print(row['transcript'])
        count += 1
    else:
        print("No transcript")

No transcript
true
No transcript
No transcript
No transcript
true
No transcript
true
No transcript
No transcript
true
No transcript
true
true
No transcript
No transcript
true
true
No transcript
No transcript
true
No transcript
true
true
No transcript
true
No transcript
true
No transcript
No transcript
No transcript
No transcript
true
No transcript
true
No transcript
true
No transcript
true
No transcript
No transcript
No transcript
No transcript
true
true
No transcript
true
No transcript
true
No transcript


In [24]:
count

21

In [48]:
transformed_data[0]['transcript']

'Mới đang cần một bạn đi rừng hoặc là một\nbạn đánh support cũng được để m đi ad.\nCó một bạn nào trong kênh chat có\nprofile uy tín không? Lên cái tên đi.\nKêu immortal á. Ờ emotal đâu ấy anh em\nnhờ?\nĐây đây đây.\n Anh M ơi em tới đây\n rồi mời\nanh đi mô tơi. Alo alo alo alo alo alo.\nỔn không anh m?\n Ổn em ơi. Mic ổn mic ổn\nnhưng mà tình hình của anh thì không ổn\nlắm đâu nha.\n[âm nhạc]\nGiật ghê quá đó ba g luôn.\n[âm nhạc]\nChết\nngon.\nPH đó tao có làm gì đâu mà ban thỉnh\nkhen tao hay ta. Hay là mình hay tới cái\nmức mà mình cũng [\xa0__\xa0] nhận ra bản thân\nmình hay luôn anh em.\nHương này nhỉnh hề quá anh em ha.\nKhổ quên quá khổ em.\n[âm nhạc]\nMá vậy mà vẫn dính là sao trời? Con\ntướng đúng rác luôn á. Có chơ game kìa.\nChừng bó ra nhanh m\nkiếm á hếtl có gợ á. Em ơi bên game này\nbên\n má nó đi cái đường này [\xa0__\xa0] thấy mà\nsao con Nunu nó [\xa0__\xa0] có l mà ban hình nãy\ngiờ 5 phút [\xa0__\xa0] ra nó găng là sao? Lê Hữu\nLinh version 2 à?\n[âm nhạc]\nDa\nqu 